In [ ]:
#!/usr/bin/env python3
# Quantus 0.6.0 evaluation for GLOBAL tabular attributions

import warnings
import logging
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn
import quantus

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Config
RANDOM_STATE = 42
DATASET_NAME = "mi_complications"
MODELS = ["logistic", "svm", "random_forest"]

DATA_ROOT = Path("../../datasets")
ATTR_ROOT = Path("./output/feature_valuation_uci_paper_eval_multi_master_corrected_FINAL")
OUT_ROOT = Path("./output/quantus_results_global_fixed")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_NUM_FEATURES = 111


# Data loader
def load_mi_complications(binary_let_is: bool = True):
    path = DATA_ROOT / "MI.data"
    df = pd.read_csv(path, header=None, na_values="?")

    X_df = df.iloc[:, 1:112]  # 111 features
    X = SimpleImputer(strategy="most_frequent").fit_transform(X_df)

    let_is = df.iloc[:, 123].astype(int).values
    y = (let_is != 0).astype(int) if binary_let_is else let_is

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    return X_train, X_test, y_train, y_test


# Model factory
def get_model(name: str):
    if name == "logistic":
        return LogisticRegression(max_iter=5000, solver="lbfgs", random_state=RANDOM_STATE)
    if name == "svm":
        return SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
    if name == "random_forest":
        return RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    raise ValueError(f"Unknown model: {name}")


# Torch wrapper - handles both 2d and 4d input
class TorchSklearnProbModel(nn.Module):
    """
    Robust wrapper that handles:
    - (B, D) - standard tabular
    - (B, 1, 1, D) - pseudo-image format
    - (B*K, 1, 1, D) - batched perturbations from Infidelity
    """
    def __init__(self, sk_model):
        super().__init__()
        self.sk_model = sk_model

    def forward(self, x):
        x_np = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

        # Flatten to (N, D) regardless of input shape
        original_shape = x_np.shape
        if x_np.ndim == 4:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 2:
            pass
        else:
            x_np = x_np.reshape(x_np.shape[0], -1)

        proba = self.sk_model.predict_proba(x_np)
        return torch.tensor(proba, dtype=torch.float32)


# Tabular -> pseudo image (channel first)
def to_pseudo_image(X_2d: np.ndarray) -> np.ndarray:
    """Convert (N, D) -> (N, 1, 1, D) for channel_first=True"""
    return X_2d.astype(np.float32)[:, None, None, :]


def phi_to_pseudo_image(A_2d: np.ndarray) -> np.ndarray:
    """Convert (N, D) -> (N, 1, 1, D) for channel_first=True"""
    return A_2d.astype(np.float32)[:, None, None, :]


# Custom perturbation function
def perturb_tabular_baseline_by_indices(
    arr: np.ndarray,
    indices,
    indexed_axes=None,
    perturb_baseline="mean",
    **kwargs,
) -> np.ndarray:
    """
    Custom perturbation that handles batched indices safely.
    Works with both (B, D) and (B, 1, 1, D) formats.
    """
    x = np.asarray(arr)
    orig_shape = x.shape

    # Flatten to (B, D)
    if x.ndim == 4:
        B, _, _, D = x.shape
        x_flat = x.reshape(B, D).copy()
    elif x.ndim == 2:
        B, D = x.shape
        x_flat = x.copy()
    else:
        raise ValueError(f"Unexpected shape: {x.shape}")

    # Compute baseline
    if isinstance(perturb_baseline, str):
        if perturb_baseline == "mean":
            baseline_vec = np.mean(x_flat, axis=0)
        elif perturb_baseline == "zero":
            baseline_vec = np.zeros(D, dtype=x_flat.dtype)
        else:
            baseline_vec = np.mean(x_flat, axis=0)
    else:
        pb = np.asarray(perturb_baseline)
        if pb.ndim == 0:
            baseline_vec = np.full(D, pb.item(), dtype=x_flat.dtype)
        else:
            baseline_vec = pb.reshape(-1).astype(x_flat.dtype)
            if baseline_vec.shape[0] != D:
                baseline_vec = np.mean(x_flat, axis=0)

    # Handle indices
    idx = np.asarray(indices)
    if idx.ndim == 0:
        idx = np.full((B, 1), int(idx))
    elif idx.ndim == 1:
        if idx.shape[0] == B:
            idx = idx.reshape(B, 1)
        else:
            idx = np.tile(idx.reshape(1, -1), (B, 1))
    else:
        idx = idx.reshape(B, -1)

    # Apply perturbations
    for b in range(B):
        row_idx = idx[b]
        row_idx = row_idx[np.isfinite(row_idx)].astype(int)
        row_idx = row_idx[(row_idx >= 0) & (row_idx < D)]
        if row_idx.size > 0:
            x_flat[b, row_idx] = baseline_vec[row_idx]

    # Reshape back to original format
    if orig_shape == x_flat.shape:
        return x_flat
    else:
        return x_flat.reshape(orig_shape)


# Load attributions
def load_attributions(dataset: str, model: str) -> Dict[str, np.ndarray]:
    base = ATTR_ROOT / f"{dataset}_{model}_results"
    methods = ["gately", "least_core_mc", "mc_shapley"]

    out: Dict[str, np.ndarray] = {}
    for m in methods:
        p = base / f"{dataset}_{model}_feature_valuations_{m}.csv"
        if not p.exists():
            logger.warning(f"Missing: {p}")
            continue

        df = pd.read_csv(p)
        vals = df["value"].values.astype(float)
        if len(vals) != EXPECTED_NUM_FEATURES:
            logger.warning(f"{model}/{m}: expected {EXPECTED_NUM_FEATURES}, got {len(vals)}")
        out[m] = vals
        logger.info(f"Loaded {model}/{m} ({len(vals)} features)")

    if not out:
        raise ValueError(f"No attributions found for {dataset}/{model}")

    return out


# Metrics - reduced set (skip infidelity if it fails)
def get_metrics():
    """
    All 7 metrics, but we'll handle Infidelity with extra care
    """
    return {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=100,
            subset_size=20,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "selectivity": quantus.Selectivity(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "monotonicity": quantus.Monotonicity(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        # Infidelity - may fail due to Quantus internal reshape issues
        "infidelity": quantus.Infidelity(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
            n_perturb_samples=50,  # Reduced from 100 to avoid memory issues
            normalise=True,
        ),
        "sparseness": quantus.Sparseness(),
        "complexity": quantus.Complexity(),
    }


# Alternative: compute aopc instead of infidelity
def compute_aopc_manual(model, X_test, y_test, phi_global, top_k_list=[5, 10, 20, 30]):
    """
    Manual AOPC computation as fallback if Infidelity fails.
    AOPC = Area Over Perturbation Curve
    """
    baseline = np.mean(X_test, axis=0)
    base_proba = model.predict_proba(X_test)
    base_scores = base_proba[np.arange(len(y_test)), y_test]

    # Sort features by importance
    sorted_idx = np.argsort(-np.abs(phi_global))

    aopc_scores = []
    for k in top_k_list:
        X_perturbed = X_test.copy()
        X_perturbed[:, sorted_idx[:k]] = baseline[sorted_idx[:k]]

        perturbed_proba = model.predict_proba(X_perturbed)
        perturbed_scores = perturbed_proba[np.arange(len(y_test)), y_test]

        score_drop = np.mean(base_scores - perturbed_scores)
        aopc_scores.append(score_drop)

    return np.mean(aopc_scores)


# Eval
def evaluate_one_model(model_name: str, X_train_raw, X_test_raw, y_train, y_test):
    logger.info("\n" + "=" * 70)
    logger.info(f"Evaluating GLOBAL Quantus metrics | {model_name}")
    logger.info("=" * 70)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    sk_model = get_model(model_name)
    sk_model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, sk_model.predict(X_train))
    test_acc = accuracy_score(y_test, sk_model.predict(X_test))
    logger.info(f"Train acc: {train_acc:.4f}")
    logger.info(f"Test  acc: {test_acc:.4f}")

    torch_model = TorchSklearnProbModel(sk_model).eval()

    Xq = to_pseudo_image(X_test)
    yq = y_test.astype(int)

    attrs = load_attributions(DATASET_NAME, model_name)
    metrics = get_metrics()

    rows = []
    for method, phi in attrs.items():
        logger.info(f"\n--- Method: {method} ---")

        A2d = np.tile(phi[None, :], (X_test.shape[0], 1))
        Aq = phi_to_pseudo_image(A2d)

        row = {
            "dataset": DATASET_NAME,
            "model": model_name,
            "method": method,
            "train_acc": train_acc,
            "test_acc": test_acc,
        }

        for mname, metric in metrics.items():
            try:
                logger.info(f"  Computing {mname} ...")
                scores = metric(
                    model=torch_model,
                    x_batch=Xq,
                    y_batch=yq,
                    a_batch=Aq,
                    channel_first=True,
                    softmax=False,
                )
                row[mname] = float(np.mean(scores))
                logger.info(f"    {mname} = {row[mname]:.6f}")

            except Exception as e:
                logger.warning(f"    {mname} failed: {e}")

                # If Infidelity fails, compute AOPC manually as alternative
                if mname == "infidelity":
                    logger.info(f"    Computing manual AOPC as fallback...")
                    try:
                        aopc = compute_aopc_manual(sk_model, X_test, y_test, phi)
                        row["aopc_manual"] = float(aopc)
                        logger.info(f"    aopc_manual = {aopc:.6f}")
                    except Exception as e2:
                        logger.warning(f"    Manual AOPC also failed: {e2}")
                        row["aopc_manual"] = np.nan

                row[mname] = np.nan

        rows.append(row)

    df = pd.DataFrame(rows)
    out_csv = OUT_ROOT / f"{DATASET_NAME}_{model_name}_quantus_global.csv"
    df.to_csv(out_csv, index=False)
    logger.info(f"\nSaved: {out_csv}")
    logger.info("\n" + df.to_string(index=False))


def main():
    X_train, X_test, y_train, y_test = load_mi_complications(binary_let_is=True)
    logger.info(f"Train: {X_train.shape}, Test: {X_test.shape}")
    logger.info(f"Class distribution: {np.bincount(y_train.astype(int))}")

    for m in MODELS:
        try:
            evaluate_one_model(m, X_train, X_test, y_train, y_test)
        except Exception as e:
            logger.exception(f"Failed for model {m}: {e}")

    logger.info("\n" + "=" * 70)
    logger.info("All evaluations complete!")
    logger.info("=" * 70)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# Quantus 0.6.0 evaluation for tabular attributions
# Extended version with instance-level explanations for LLE/MS
# Total metrics reported (10):
#   1) faithfulness_corr (global)
#   2) faithfulness_estimate (global)
#   3) selectivity (global)
#   4) monotonicity (global)
#   5) sparseness (global)
#   6) complexity (global)
#   7) aopc_manual (global)
#   8) pixel_flipping (global)
#   9) local_lipschitz_estimate (instance-level)
#   10) max_sensitivity (instance-level)

import warnings
import logging
from pathlib import Path
from typing import Dict, List, Callable

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn
import quantus

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Config
RANDOM_STATE = 42
DATASET_NAME = "mi_complications"
MODELS = ["logistic", "svm", "random_forest"]

DATA_ROOT = Path("../../datasets")
ATTR_ROOT = Path("./output/feature_valuation_uci_paper_eval_multi_master_corrected_FINAL")
OUT_ROOT = Path("./output/quantus_results_global_EXTENDED")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_NUM_FEATURES = 111

# AOPC config
AOPC_TOPK_LIST = [5, 10, 20, 30]

# Instance-level explanation config
NUM_SAMPLES_FOR_INSTANCE_METRICS = 50  # Compute on subset for efficiency


# Data loader
def load_mi_complications(binary_let_is: bool = True):
    path = DATA_ROOT / "MI.data"
    df = pd.read_csv(path, header=None, na_values="?")

    X_df = df.iloc[:, 1:112]  # 111 features
    X = SimpleImputer(strategy="most_frequent").fit_transform(X_df)

    let_is = df.iloc[:, 123].astype(int).values
    y = (let_is != 0).astype(int) if binary_let_is else let_is

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    return X_train, X_test, y_train, y_test


# Model factory
def get_model(name: str):
    if name == "logistic":
        return LogisticRegression(max_iter=5000, solver="lbfgs", random_state=RANDOM_STATE)
    if name == "svm":
        return SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
    if name == "random_forest":
        return RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    raise ValueError(f"Unknown model: {name}")


# Torch wrapper
class TorchSklearnProbModel(nn.Module):
    """Wrap sklearn predict_proba as torch.nn.Module"""
    def __init__(self, sk_model):
        super().__init__()
        self.sk_model = sk_model

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = np.asarray(x)

        # Flatten to (N, D)
        x_np = x_np.reshape(x_np.shape[0], -1)
        proba = self.sk_model.predict_proba(x_np)
        return torch.tensor(proba, dtype=torch.float32)


# Tabular -> pseudo image
def to_pseudo_image(X_2d: np.ndarray) -> np.ndarray:
    """(N, D) -> (N, 1, 1, D)"""
    return X_2d.astype(np.float32)[:, None, None, :]


def phi_to_pseudo_image(A_2d: np.ndarray) -> np.ndarray:
    """(N, D) -> (N, 1, 1, D)"""
    return A_2d.astype(np.float32)[:, None, None, :]


# Custom perturb func
def perturb_tabular_baseline_by_indices(
    arr: np.ndarray,
    indices,
    indexed_axes=None,
    perturb_baseline="mean",
    **kwargs,
) -> np.ndarray:
    """Baseline replacement for tabular data"""
    x = np.asarray(arr)
    orig_shape = x.shape

    # Flatten to (B, D)
    x_flat = x.reshape(x.shape[0], -1).copy()
    B, D = x_flat.shape

    # baseline vector
    if isinstance(perturb_baseline, str):
        if perturb_baseline == "mean":
            baseline_vec = np.mean(x_flat, axis=0)
        elif perturb_baseline == "zero":
            baseline_vec = np.zeros(D, dtype=x_flat.dtype)
        else:
            baseline_vec = np.mean(x_flat, axis=0)
    else:
        pb = np.asarray(perturb_baseline)
        if pb.ndim == 0:
            baseline_vec = np.full(D, pb.item(), dtype=x_flat.dtype)
        else:
            baseline_vec = pb.reshape(-1).astype(x_flat.dtype)
            if baseline_vec.shape[0] != D:
                baseline_vec = np.mean(x_flat, axis=0)

    # indices -> (B, k)
    idx = np.asarray(indices)
    if idx.ndim == 0:
        idx = np.full((B, 1), int(idx))
    elif idx.ndim == 1:
        if idx.shape[0] == B:
            idx = idx.reshape(B, 1)
        else:
            idx = np.tile(idx.reshape(1, -1), (B, 1))
    else:
        idx = idx.reshape(B, -1)

    for b in range(B):
        row_idx = idx[b]
        row_idx = row_idx[np.isfinite(row_idx)].astype(int)
        row_idx = row_idx[(row_idx >= 0) & (row_idx < D)]
        if row_idx.size > 0:
            x_flat[b, row_idx] = baseline_vec[row_idx]

    return x_flat.reshape(orig_shape)


# Load global attributions
def load_attributions(dataset: str, model: str) -> Dict[str, np.ndarray]:
    base = ATTR_ROOT / f"{dataset}_{model}_results"
    methods = ["gately", "least_core_mc", "mc_shapley"]

    out: Dict[str, np.ndarray] = {}
    for m in methods:
        p = base / f"{dataset}_{model}_feature_valuations_{m}.csv"
        if not p.exists():
            logger.warning(f"Missing: {p}")
            continue

        df = pd.read_csv(p)
        vals = df["value"].values.astype(float)

        if len(vals) != EXPECTED_NUM_FEATURES:
            logger.warning(f"{model}/{m}: expected {EXPECTED_NUM_FEATURES}, got {len(vals)}")

        out[m] = vals
        logger.info(f"Loaded {model}/{m} ({len(vals)} features)")

    if not out:
        raise ValueError(f"No attributions found for {dataset}/{model}")

    return out


# Instance-level explanation generator
def compute_instance_shapley(
    sk_model,
    X_train: np.ndarray,
    y_train: np.ndarray,
    x_instance: np.ndarray,
    n_permutations: int = 100,
) -> np.ndarray:
    """
    Compute instance-level Shapley values for a single sample.
    Uses permutation-based approximation.

    Returns:
        shapley_values: (n_features,) array
    """
    n_features = X_train.shape[1]
    shapley_values = np.zeros(n_features)

    # Get baseline prediction (using training mean)
    baseline = np.mean(X_train, axis=0)

    for _ in range(n_permutations):
        # Random feature permutation
        perm = np.random.permutation(n_features)

        # Build coalition incrementally
        x_partial = baseline.copy()
        prev_pred = sk_model.predict_proba(x_partial.reshape(1, -1))[0]

        for feat_idx in perm:
            # Add feature to coalition
            x_partial[feat_idx] = x_instance[feat_idx]
            curr_pred = sk_model.predict_proba(x_partial.reshape(1, -1))[0]

            # Marginal contribution
            marginal = curr_pred - prev_pred

            # Accumulate (use probability of true class)
            # For simplicity, use max probability class
            shapley_values[feat_idx] += marginal[np.argmax(curr_pred)]

            prev_pred = curr_pred

    # Average over permutations
    shapley_values /= n_permutations

    return shapley_values


def create_quantus_explain_func_for_method(
    method: str,
    sk_model,
    X_train: np.ndarray,
    y_train: np.ndarray,
    global_phi: np.ndarray,
):
    """
    Create a Quantus-compatible explain function.

    Quantus will call this as: explain_func(model, inputs, targets, **kwargs)
    and expects it to return explanations in the same shape as inputs.
    """

    def explain_func_wrapper(model, inputs, targets, **kwargs):
        """
        Quantus-compatible explanation function.

        Args:
            model: PyTorch model (we ignore this, use sk_model)
            inputs: np.ndarray or torch.Tensor, shape (B, 1, 1, D)
            targets: np.ndarray or torch.Tensor, shape (B,)
            **kwargs: Additional arguments (ignored)

        Returns:
            explanations: np.ndarray, shape (B, 1, 1, D)
        """
        # Convert to numpy
        if torch.is_tensor(inputs):
            X = inputs.cpu().numpy()
        else:
            X = np.asarray(inputs)

        # Flatten
        X_flat = X.reshape(X.shape[0], -1)
        B, D = X_flat.shape

        explanations = np.zeros((B, D), dtype=np.float32)

        if "shapley" in method.lower():
            # Compute instance-level Shapley
            for i in range(B):
                try:
                    explanations[i] = compute_instance_shapley(
                        sk_model, X_train, y_train, X_flat[i],
                        n_permutations=50
                    )
                except Exception as e:
                    logger.warning(f"Shapley computation failed for sample {i}: {e}")
                    explanations[i] = global_phi
        else:
            # Global methods: use global attribution
            explanations = np.tile(global_phi, (B, 1))

        # Reshape back to input shape
        return explanations.reshape(X.shape).astype(np.float32)

    return explain_func_wrapper


# Metrics
def get_global_metrics():
    """Metrics that work with pre-computed global attributions"""
    return {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=100,
            subset_size=20,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "selectivity": quantus.Selectivity(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "monotonicity": quantus.Monotonicity(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "sparseness": quantus.Sparseness(),
        "complexity": quantus.Complexity(),
        "pixel_flipping": quantus.PixelFlipping(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
    }


def get_instance_metrics():
    """Metrics that require instance-level explanation generation"""

    # Define norm functions (Quantus expects callables)
    # These are called as distance functions: norm(a, b) for LLE
    # and as single-input norms: norm(a) for MS

    def l2_norm(a, b=None, **kwargs):
        """L2 (Euclidean) norm - works for both distance and magnitude"""
        a_flat = np.asarray(a).flatten()
        if b is not None:
            b_flat = np.asarray(b).flatten()
            return np.linalg.norm(a_flat - b_flat, ord=2)
        return np.linalg.norm(a_flat, ord=2)

    return {
        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=10,
            perturb_std=0.1,
            perturb_mean=0.0,
            norm_numerator=l2_norm,
            norm_denominator=l2_norm,
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=10,
            lower_bound=0.2,
            norm_numerator=l2_norm,
            norm_denominator=l2_norm,
            perturb_func=perturb_tabular_baseline_by_indices,
        ),
    }


# Manual aopc
def compute_aopc_manual(
    sk_model,
    X_test_2d: np.ndarray,
    y_test: np.ndarray,
    phi_global: np.ndarray,
    top_k_list: List[int],
) -> float:
    """AOPC: mean drop in P(y_true) after perturbing top-k features"""
    baseline = np.mean(X_test_2d, axis=0)
    base_proba = sk_model.predict_proba(X_test_2d)
    base_scores = base_proba[np.arange(len(y_test)), y_test]

    sorted_idx = np.argsort(-np.abs(phi_global))

    drops = []
    for k in top_k_list:
        k = min(k, X_test_2d.shape[1])
        Xp = X_test_2d.copy()
        Xp[:, sorted_idx[:k]] = baseline[sorted_idx[:k]]

        pp = sk_model.predict_proba(Xp)
        ps = pp[np.arange(len(y_test)), y_test]

        drops.append(np.mean(base_scores - ps))

    return float(np.mean(drops))


# Evaluation
def evaluate_one_model(model_name: str, X_train_raw, X_test_raw, y_train, y_test):
    logger.info("\n" + "=" * 70)
    logger.info(f"Evaluating EXTENDED Quantus metrics | {model_name}")
    logger.info("=" * 70)

    # Scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    # Train sklearn model
    sk_model = get_model(model_name)
    sk_model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, sk_model.predict(X_train))
    test_acc = accuracy_score(y_test, sk_model.predict(X_test))
    logger.info(f"Train acc: {train_acc:.4f}")
    logger.info(f"Test  acc: {test_acc:.4f}")

    # Torch wrapper
    torch_model = TorchSklearnProbModel(sk_model).eval()

    # Full test set for global metrics
    Xq_full = to_pseudo_image(X_test)
    yq_full = y_test.astype(int)

    # Subset for instance-level metrics (for efficiency)
    sample_indices = np.random.choice(
        len(X_test),
        size=min(NUM_SAMPLES_FOR_INSTANCE_METRICS, len(X_test)),
        replace=False
    )
    X_test_subset = X_test[sample_indices]
    y_test_subset = y_test[sample_indices]
    Xq_subset = to_pseudo_image(X_test_subset)
    yq_subset = y_test_subset.astype(int)

    attrs = load_attributions(DATASET_NAME, model_name)
    global_metrics = get_global_metrics()

    rows = []
    for method, phi in attrs.items():
        logger.info(f"\n--- Method: {method} ---")

        # Global attribution (repeated per sample)
        A2d_full = np.tile(phi[None, :], (X_test.shape[0], 1))
        Aq_full = phi_to_pseudo_image(A2d_full)

        row = {
            "dataset": DATASET_NAME,
            "model": model_name,
            "method": method,
            "train_acc": float(train_acc),
            "test_acc": float(test_acc),
        }

        for mname, metric in global_metrics.items():
            logger.info(f"  Computing {mname} ...")
            try:
                scores = metric(
                    model=torch_model,
                    x_batch=Xq_full,
                    y_batch=yq_full,
                    a_batch=Aq_full,
                    channel_first=True,
                    softmax=False,
                    batch_size=64,
                )
                row[mname] = float(np.mean(scores))
                logger.info(f"    {mname} = {row[mname]:.6f}")
            except Exception as e:
                logger.warning(f"    {mname} FAILED: {e}")
                row[mname] = np.nan

        logger.info("  Computing aopc_manual ...")
        row["aopc_manual"] = compute_aopc_manual(
            sk_model=sk_model,
            X_test_2d=X_test,
            y_test=y_test.astype(int),
            phi_global=phi,
            top_k_list=AOPC_TOPK_LIST,
        )
        logger.info(f"    aopc_manual = {row['aopc_manual']:.6f}")

        logger.info(f"  Computing instance-level metrics on {len(X_test_subset)} samples...")

        # Create explain function for this method
        explain_func = create_quantus_explain_func_for_method(
            method=method,
            sk_model=sk_model,
            X_train=X_train,
            y_train=y_train,
            global_phi=phi,
        )

        instance_metrics = get_instance_metrics()

        for mname, metric in instance_metrics.items():
            logger.info(f"  Computing {mname} ...")
            try:
                # Test explain_func first
                logger.info(f"    Testing explain_func on 1 sample...")
                test_exp = explain_func(torch_model, Xq_subset[:1], yq_subset[:1])
                logger.info(f"    explain_func works! Output shape: {test_exp.shape}")

                # Call metric with explain_func and explain_func_kwargs
                scores = metric(
                    model=torch_model,
                    x_batch=Xq_subset,
                    y_batch=yq_subset,
                    a_batch=None,  # Let explain_func generate
                    channel_first=True,
                    softmax=False,
                    batch_size=16,
                    explain_func=explain_func,
                    explain_func_kwargs={},  # Empty dict, not needed
                )
                row[mname] = float(np.mean(scores))
                logger.info(f"    {mname} = {row[mname]:.6f}")
            except Exception as e:
                import traceback
                logger.error(f"    {mname} FAILED:")
                logger.error(traceback.format_exc())
                row[mname] = np.nan

        rows.append(row)

    df = pd.DataFrame(rows)
    out_csv = OUT_ROOT / f"{DATASET_NAME}_{model_name}_quantus_extended.csv"
    df.to_csv(out_csv, index=False)

    logger.info(f"\nSaved: {out_csv}")
    logger.info("\n" + df.to_string(index=False))


def main():
    X_train, X_test, y_train, y_test = load_mi_complications(binary_let_is=True)
    logger.info(f"Train: {X_train.shape}, Test: {X_test.shape}")
    logger.info(f"Class distribution: {np.bincount(y_train.astype(int))}")

    for m in MODELS:
        try:
            evaluate_one_model(m, X_train, X_test, y_train, y_test)
        except Exception as e:
            logger.exception(f"Failed for model {m}: {e}")

    logger.info("\n" + "=" * 70)
    logger.info("All evaluations complete!")
    logger.info("=" * 70)


if __name__ == "__main__":
    main()